# YOLO03 SizeBalanced 학습

기존 YOLO best_solo.pt(e26, SHA c5c85272700c…)에서 시작합니다. Seed 2026, crop 32, Eu 크기 균형. best50는 결합 비율 50%의 선택 기준입니다.


## 설정


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/KoMaP2026')
DATA_ROOT=Path('/content/komap_eu_upgrade_data')
CURRENT_YOLO_RUN_NAME=None
DINO_RUN_NAME='dinov3_hard_ablation_control_01'
EXPECTED_DINO_SHA='1361e35f7331e417704b216bb0253ce221c8d69f58b44e7bce0bd05dcdae17b3'
BASE_RUN_NAMES=['specialist_primary_fast_01','specialist_al3ni_fast_01',
                'specialist_eutectic_fast_01','specialist_al_fast_01']
EU_RUN_NAME='advanced_eu_balanced_ram_01'
BASE_WEIGHTS=[1.,1.,1.,1.]
ABSENT_POLICY='ignore'
NUM_WORKERS=2
RESUME=True
RUN_PREFIX='yolo_eu_03_sizebalanced_20261011'
DECODER_P1=False
DETAIL_CHANNELS=16
EU_BALANCED=True
SEEDS=[2026]
EPOCHS=60
BATCH_SIZE=4
PATCHES_PER_IMAGE=32
EVAL_EVERY=2


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
import subprocess
subprocess.check_call([sys.executable, str(CODE_ROOT/'komap_submission_environment.py'), '--install'])


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
CODE_DIR = CODE_ROOT / '.'
sys.dont_write_bytecode = True

for _module_name in list(sys.modules):
    if _module_name.startswith('komap_'):
        del sys.modules[_module_name]
sys.path[:] = [p for p in sys.path if p not in (str(CODE_ROOT), str(CODE_ROOT/'legacy_yolo_initial'))]
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
print('제출 코드:', CODE_DIR)


## 데이터·초기 모델


In [ ]:
import json,numpy as np,pandas as pd,torch
import komap_specialist_data as kd
import komap_phase_fusion as pf
import komap_nieu_zoom_ram as kr
import komap_metal_methods_ram as mm
import komap_yolo_upgrade_support as us
for module in (kd,pf,kr,mm,us):
    if Path(module.__file__).resolve().parent!=CODE_DIR.resolve():raise RuntimeError('잘못된 모듈 경로')
RAM,DATA_REPORT=kr.prepare_ram(PROJECT_ROOT,DATA_ROOT)
DINO_INFO=kr.source_from_completed_control(PROJECT_ROOT,DATA_REPORT,DINO_RUN_NAME)
if DINO_INFO['sha256']!=EXPECTED_DINO_SHA or DINO_INFO['source_epoch']!=6:
    raise ValueError('확인된 DINO control e006과 다릅니다.')
BASE_EXPORTS=pf.resolve_baseline(PROJECT_ROOT,BASE_RUN_NAMES)
EU_EXPORT=pf.load_export(PROJECT_ROOT/'runs'/EU_RUN_NAME)
OLD_BASELINE=mm.prepare_current_baseline(RAM,DATA_REPORT,DINO_INFO,BASE_EXPORTS,EU_EXPORT,BASE_WEIGHTS)
if abs(OLD_BASELINE['validation']['miou_ignore']-us.OLD_MIOU)>1e-8:
    raise ValueError('이전 조합 0.795970520243이 재현되지 않습니다.')
CURRENT_INFO=us.resolve_current_yolo(PROJECT_ROOT,RAM,DATA_REPORT,CURRENT_YOLO_RUN_NAME)
RAW_YOLO=us.current_yolo_context(RAM,DATA_REPORT,CURRENT_INFO)
CURRENT_REFERENCE=us.current_reference(RAM,RAW_YOLO,OLD_BASELINE)
display(pd.DataFrame([{'candidate':'현재 조합',**CURRENT_REFERENCE['validation']['class_iou_ignore'],
                     'miou_ignore':CURRENT_REFERENCE['validation']['miou_ignore'],
                     'miou_zero':CURRENT_REFERENCE['validation']['miou_zero']}]))


## 학습·검증


In [ ]:
SEED_RUN_DIRS=[]
for seed in SEEDS:
    run_dir=PROJECT_ROOT/'runs'/f'{RUN_PREFIX}_seed{seed}'
    C=us.experiment_config(CURRENT_INFO,seed=seed,epochs=EPOCHS,batch=BATCH_SIZE,
        patches_per_image=PATCHES_PER_IMAGE,eval_every=EVAL_EVERY,absent_policy=ABSENT_POLICY,
        decoder_p1=DECODER_P1,detail_channels=DETAIL_CHANNELS,eu_component_balanced=EU_BALANCED)
    print('새 실험:',run_dir,'P1=',C.decoder_p1,'detail=',C.detail_channels,'Eu 균형=',C.eu_component_balanced)
    mm.train(RAM,DATA_REPORT,run_dir,C,OLD_BASELINE,num_workers=NUM_WORKERS,resume=RESUME,
             initial_checkpoint=CURRENT_INFO['path'])

    one_export=mm.export_candidates(RAM,DATA_REPORT,run_dir,OLD_BASELINE,
                                     current_reference=CURRENT_REFERENCE)
    SEED_RUN_DIRS.append(run_dir)
    display(pd.read_csv(one_export/'candidate_ranking.csv'))
    print('각 seed best50:',run_dir/'best50.pt')
EXPORT_DIR=us.export_seed_ensemble(RAM,DATA_REPORT,PROJECT_ROOT/'runs'/(RUN_PREFIX+'_seed_comparison'),
    SEED_RUN_DIRS,OLD_BASELINE,CURRENT_REFERENCE,ABSENT_POLICY)


## 결과


In [ ]:
display(pd.read_csv(EXPORT_DIR/'candidate_ranking.csv'))
RESULT=json.loads((EXPORT_DIR/'validation_summary.json').read_text(encoding='utf-8'))
print('선택:',RESULT['selected_candidate'],'valid mIoU:',RESULT['selected']['miou_'+ABSENT_POLICY])
print('원본 해상도 test10 마스크:',EXPORT_DIR/'selected_test_masks.zip')
print('후속 분석 파일:',EXPORT_DIR/'analysis_bundle.zip')


## 결과 다운로드


In [ ]:
DOWNLOAD_RESULTS=False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(EXPORT_DIR/'analysis_bundle.zip'))
    files.download(str(EXPORT_DIR/'selected_test_masks.zip'))
